# 08 - Funding & Open Interest

Crowded positioning → reversal signal.


## Setup

In [1]:
# --- setup --------------------------------------------------------
import sys, json
sys.path.insert(0, r"D:\\Projects\\done\\eth_bot\\eth_bot_v2")
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
np.random.seed(42)

SYMBOL = "ETHUSDT"
OUT = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "reports" / "forecasts"
OUT.mkdir(parents=True, exist_ok=True)
DB = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "database" / "market.db"

def sql(q, **p):
    with sqlite3.connect(DB) as c:
        return pd.read_sql_query(q, c, params=p)

from database.config.api.binance import get_klines

def fetch(symbol, interval, limit=500):
    raw = get_klines(symbol, interval=interval, limit=limit)
    df = pd.DataFrame(raw, columns=["open_time","open","high","low","close",
        "volume","close_time","quote_volume","trades","taker_base","taker_quote","ignore"])
    df["date"] = pd.to_datetime(df["open_time"], unit="ms", utc=True).dt.tz_convert(None)
    for c in ("open","high","low","close","volume"):
        df[c] = df[c].astype(float)
    return df[["date","open","high","low","close","volume"]].set_index("date").iloc[:-1]

def action_block(name, action, reason, price=None, entry=None, stop=None,
                 target=None, conf=None, extra=None):
    print("=" * 62)
    print(f"  {name}")
    print("=" * 62)
    print(f"  time   : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')} UTC")
    if price is not None: print(f"  price  : {price:,.2f}")
    if conf  is not None: print(f"  conf   : {conf:.2f}")
    print(f"  ACTION : {action}")
    print(f"  REASON : {reason}")
    if entry is not None:
        print(f"  entry  : {entry:,.2f}   stop: {stop:,.2f}   target: {target:,.2f}")
    if extra:
        for k, v in extra.items():
            print(f"  {k:<6} : {v}")
    print("=" * 62)
    return {"action": action, "reason": reason, "price": price,
            "entry": entry, "stop": stop, "target": target, "conf": conf}

## 1. Load funding + OI

In [2]:
fr = sql("SELECT ts, rate FROM funding_rate ORDER BY ts")
fr["date"] = pd.to_datetime(fr.ts, unit="ms"); fr = fr.set_index("date")["rate"]
oi = sql("SELECT ts, oi FROM open_interest ORDER BY ts")
oi["date"] = pd.to_datetime(oi.ts, unit="ms"); oi = oi.set_index("date")["oi"]
print("funding n=", len(fr), " oi n=", len(oi))

funding n= 500  oi n= 30


## 2. Current funding

In [3]:
cur_fr = float(fr.iloc[-1])
print(f"latest funding: {cur_fr*100:+.4f}% per 8h")
if cur_fr > 0.0005: print("> longs pay shorts — crowded long")
elif cur_fr < -0.0005: print("> shorts pay longs — crowded short")
else: print("> neutral")

latest funding: +0.0001% per 8h
> neutral


## 3. OI change

In [4]:
oi_chg = (oi.iloc[-1] / oi.iloc[-8] - 1) * 100
print(f"OI change (last 7d): {oi_chg:+.2f}%")

OI change (last 7d): +1.76%


## 4. Action

In [5]:
if cur_fr > 0.001 and oi_chg > 5:    action, reason = "SHORT", "crowded long — squeeze risk"
elif cur_fr < -0.001 and oi_chg > 5:  action, reason = "LONG",  "crowded short — squeeze up"
else:                                  action, reason = "FLAT",  "positioning neutral"
action_block("FUNDING / OI", action, reason,
    extra={"funding": f"{cur_fr*100:+.4f}%", "oi_7d_chg": f"{oi_chg:+.2f}%"})

  FUNDING / OI
  time   : 2026-10-04 10:44 UTC
  ACTION : FLAT
  REASON : positioning neutral
  funding : +0.0001%
  oi_7d_chg : +1.76%


{'action': 'FLAT',
 'reason': 'positioning neutral',
 'price': None,
 'entry': None,
 'stop': None,
 'target': None,
 'conf': None}